# Exploratory analysis: participation ratio alternatives

This notebook is outside the main paper workflow. Existing results and metric variants are historical/exploratory, not the authoritative paper results. Original analysis cells and outputs are retained. Some legacy sections require selective execution; this is not a verified clean-run pipeline.

The main workflow is in `notebooks/paper/`. This notebook may write legacy exports under `analysis_exports/`; the paper pipeline uses its own `outputs/paper/` directory.

Plot defaults now come from `plot_style.py`. Prior static image outputs were cleared; rerun plot cells after their prerequisites to see the shared style. Specialized heatmap scales and animations retain their own semantic encodings.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/wired").is_dir() and (p / "paper").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this repository or a notebook directory inside it.")
os.chdir(PROJECT_ROOT)  # Preserve project-relative paths when launched from a subdirectory.
print("Project root:", PROJECT_ROOT)

# Shared project plotting conventions.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from plot_style import (
    apply_style, expertise_palette, EXPERTISE_COLORS, LEVEL_LABELS,
    METRIC_LABELS, MODEL_COLORS, MODEL_MARKERS, PRIMARY, NEUTRAL,
    plot_metric_trajectories, save_figure,
)
apply_style()


# Participation-ratio analysis

Naive and bias-corrected participation ratio, matched statistical analysis, cumulative and rolling PR trajectories, explanatory visualizations, and the dual PCA–PR animation.

This notebook was separated from `big_analysis_centroid_fixed(1).ipynb`. Stale outputs were removed so results are regenerated from the current data. Run cells from top to bottom with the working directory set to the project root containing `data/wired`.

Load csvs. 1 csv- one conversation. each row, one utterance. some consecutive speaker rows.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

PROJECT_ROOT = Path.cwd()
WIRED_DIR = PROJECT_ROOT / "data" / "wired"

csv_paths = sorted(WIRED_DIR.glob("wired_*/*.csv"))

print(f"Found {len(csv_paths)} CSV files")
csv_paths[:10]

Found 105 CSV files


[PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_astro/wired_astro_16.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_12.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_13.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_14.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_15.csv'),
 PosixPath('/Users/yume/PycharmProjects/horizons_2/data/wired/wired_blockchain/wired_blockchain_

In [2]:
def parse_wired_path(path):
    """
    Example:
    data/wired/wired_astro/wired_astro_12.csv
    """
    folder_name = path.parent.name
    file_stem = path.stem

    video_id = re.sub(r"^wired_", "", folder_name)

    match = re.search(r"_(\d+)$", file_stem)
    file_number = int(match.group(1)) if match else None

    return {
        "dataset": "wired",
        "video_id": video_id,
        "file_number": file_number,
        "conversation_id": file_stem,
        "source_file": str(path.relative_to(PROJECT_ROOT))
    }

In [3]:
raw_frames = []

for path in csv_paths:
    df = pd.read_csv(path)
    metadata = parse_wired_path(path)

    # Preserve original within-file row order
    df["raw_row_id"] = np.arange(len(df))

    for column, value in metadata.items():
        df[column] = value

    raw_frames.append(df)

wired_raw = pd.concat(
    raw_frames,
    ignore_index=True,
    sort=False
)

wired_raw.shape

(6384, 10)

table with all utterances, all conversations

In [4]:
wired_raw.head()

,Sequence,Speaker,Utterance,Notes,raw_row_id,dataset,video_id,file_number,conversation_id,source_file
0,1,Speaker 2 (child),Hi.,NaN,0,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
1,2,Speaker 1 (Janna Levin),"Hi, welcome.",NaN,1,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
2,3,Speaker 1 (Janna Levin),Tell me your name.,NaN,2,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
3,4,Speaker 2 (child),Jude.,NaN,3,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv
4,5,Speaker 1 (Janna Levin),I wanted to ask you if you have ever heard of ...,NaN,4,wired,astro,12,wired_astro_12,data/wired/wired_astro/wired_astro_12.csv


In [5]:
#rename columns, minimally clean text
wired_raw = wired_raw.rename(columns={
    "Speaker": "speaker_raw",
    "Utterance": "text"
})
wired_raw["text"] = (
    wired_raw["text"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

wired_raw = wired_raw[
    wired_raw["text"].notna() &
    wired_raw["text"].ne("")
].copy()

In [6]:
level_map = {
    12: (0, "child"),
    13: (1, "teenager"),
    14: (2, "undergraduate"),
    15: (3, "graduate"),
    16: (4, "expert")
}

In [7]:
wired_raw["level"] = wired_raw["file_number"].map(
    lambda x: level_map[x][0]
)

wired_raw["level_label"] = wired_raw["file_number"].map(
    lambda x: level_map[x][1]
)

In [8]:
#order levels
level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert"
]

wired_raw["level_label"] = pd.Categorical(
    wired_raw["level_label"],
    categories=level_order,
    ordered=True
)

In [9]:
'''
speaker_mapping = {
    #astro
     "Speaker 1 (Janna Levin)": "A",
     "Speaker 2 (child)": "B",
     "Speaker 3 (teen)": "B",
    "Speaker 4 (college student)": "B",
    "Speaker 5 (grad student)": "B",
    "Speaker 6 (expert)": "B",
    #blockchain
    "Speaker 1 (Bettina Warburg)": "A",
    "Speaker 2 (Child)": "B",
    "Speaker 3 (Teen - Ian)": "B",
    "Speaker 4 (College Student)": "B",
    "Speaker 5 (Grad Student)": "B",
    "Speaker 6 (Expert)": "B",
    #crispr
    "Speaker 1 (Neville Sanjana)": "A",
    "Speaker 2 (Tegan - Child)": "B",
    "Speaker 3 (Bella - Teenager)": "B",
    "Speaker 4 (Christopher - Student)": "B",
    "Speaker 5 (Lauren Schiff)": "B",
    "Speaker 6 (Matthew Canver)": "B",
    #dimension
    "Speaker 1 (Sean Carroll PhD)": "A",
    #fractal
    "Speaker 1 (Keenan Crane Phd)": "A",
    #gravity
    "Speaker 1 (Janna Levin)": "A",
    "Speaker 2 (Bonét Sofía Kanayet)": "B",
    "Speaker 3 (Maria Teresa Furtado)": "B",
    "Speaker 4 (Lisa Chan)": "B",
    "Speaker 5 (Will Gyory)": "B",
    "Speaker 6 (Matthew Kleban)": "B",
    #hacking
    "Speaker 1 (Samy Kamkar)": "A",
    #infinity
    "Speaker 1 (Emily Riehl)": "A",
    #internet
    "Speaker 1 (Jim Kurose)": "A",
    #laser
    "Speaker 1 (Donna Strickland)": "A",
    "Speaker 2 (Harmoni - Child)": "B",
    "Speaker 3 (Eli Kaplan - Teenager)": "B",
    "Speaker 4 (Caitlin - Student)": "B",
    "Speaker 5 (Aditya - Grad Student)": "B",
    "Speaker 6 (Mike Campbell)": "B",
    #machine
    "Speaker 1 (Hilary Mason)": "A",
    #memory
    "Speaker 1 (Daphna Shohamy)": "A",
    #moravec
    "Speaker 1 (Chelsea Finn)": "A",
    #nano
    "Speaker 1 (Dr. George S. Tulevski)": "A",
    #neuro
    "Speaker 1 (Dr. Bobby Kasthuri)": "A",
    "Speaker 2 (child - Daniel)": "B",
    "Speaker 3 (Jabez Griggs)": "B",
    "Speaker 4 (Elena Dowling)": "B",
    "Speaker 5 (Mala Ananth)": "B",
    "Speaker 6 (Russell Hanson)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",
    "Speaker 3 (teen)": "B",


}
'''
is_expert = wired_raw["speaker_raw"].str.contains(
    r"\bSpeaker\s*1\b",
    case=False,
    na=False
)

wired_raw["speaker"] = np.where(is_expert, "A", "B")
wired_raw["speaker_role"] = np.where(
    is_expert,
    "expert",
    "partner"
)




wired_raw["speaker_role"] = wired_raw["speaker"].map({
    "A": "expert",
    "B": "partner"
})

# Utterance embeddings

## utterances table -> embed utterances

In [10]:
import re
import numpy as np
import pandas as pd


def count_words(text):
    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(text),
        )
    )


def create_utterances_table(wired_raw):
    """
    Create one canonical row per original transcript utterance.

    The function does not concatenate adjacent utterances.
    """

    utterances = (
        wired_raw
        .copy()
        .reset_index(drop=False)
        .rename(columns={"index": "_original_dataframe_row"})
    )

    # Accommodate either the original CSV column names or the
    # standardized names already present in wired_raw.
    rename_map = {}

    aliases = {
        "Sequence": "sequence",
        "Utterance": "text",
        "Speaker": "speaker_raw",
        "Notes": "notes",
    }

    for old_name, new_name in aliases.items():
        if (
            old_name in utterances.columns
            and new_name not in utterances.columns
        ):
            rename_map[old_name] = new_name

    utterances = utterances.rename(columns=rename_map)

    # If canonical speaker has not already been created, use
    # the original speaker label.
    if "speaker" not in utterances.columns:
        utterances["speaker"] = utterances["speaker_raw"]

    # Stable source order.
    if "sequence" not in utterances.columns:
        utterances["sequence"] = (
            utterances
            .groupby(
                ["dataset", "conversation_id"],
                sort=False,
            )
            .cumcount()
            .add(1)
        )

    utterances["_sequence_numeric"] = pd.to_numeric(
        utterances["sequence"],
        errors="coerce",
    )

    utterances["_sequence_numeric"] = (
        utterances["_sequence_numeric"]
        .fillna(utterances["_original_dataframe_row"])
    )

    conversation_keys = [
        "dataset",
        "conversation_id",
    ]

    sort_columns = [
        *conversation_keys,
        "_sequence_numeric",
        "_original_dataframe_row",
    ]

    utterances = (
        utterances
        .sort_values(
            sort_columns,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    # Clean text without altering its linguistic contents.
    utterances["text"] = (
        utterances["text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    if utterances["text"].eq("").any():
        empty_rows = utterances.loc[
            utterances["text"].eq(""),
            [
                *conversation_keys,
                "sequence",
            ],
        ]

        raise ValueError(
            "Empty utterance text found:\n"
            + empty_rows.to_string(index=False)
        )

    # Reconstruct turn membership without concatenating anything.
    # A new turn begins whenever the speaker changes.
    speaker_changed = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )["speaker"]
        .transform(
            lambda speakers:
                speakers.ne(speakers.shift())
        )
    )

    utterances["turn_id"] = (
        speaker_changed
        .astype(int)
        .groupby(
            [
                utterances[column]
                for column in conversation_keys
            ]
        )
        .cumsum()
    )

    # Position of the utterance inside its speaker turn.
    utterances["utterance_in_turn"] = (
        utterances
        .groupby(
            [
                *conversation_keys,
                "turn_id",
            ],
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    # Position within the whole conversation.
    utterances["utterance_number"] = (
        utterances
        .groupby(
            conversation_keys,
            sort=False,
        )
        .cumcount()
        .add(1)
    )

    utterances["utterance_id"] = (
        utterances["dataset"].astype(str)
        + "::"
        + utterances["conversation_id"].astype(str)
        + "::utterance_"
        + utterances["utterance_number"]
            .astype(str)
            .str.zfill(3)
    )

    utterances["n_words"] = (
        utterances["text"]
        .map(count_words)
        .astype(int)
    )

    # Keep all utterances in the canonical table. This flag only
    # determines the primary geometry subset later.
    utterances["include_geometry_primary"] = (
        utterances["n_words"] >= 5
    )

    # This index will correspond exactly to rows of E_utterances.
    utterances["embedding_idx"] = np.arange(
        len(utterances),
        dtype=int,
    )

    preferred_columns = [
        "dataset",
        "video_id",
        "conversation_id",
        "file_number",
        "level",
        "level_label",
        "utterance_id",
        "utterance_number",
        "turn_id",
        "utterance_in_turn",
        "sequence",
        "normalized_time",
        "speaker",
        "speaker_raw",
        "speaker_role",
        "text",
        "n_words",
        "include_geometry_primary",
        "embedding_idx",
        "notes",
        "source_file",
        "_original_dataframe_row",
    ]

    # Retain only columns that actually exist in this dataset.
    output_columns = [
        column
        for column in preferred_columns
        if column in utterances.columns
    ]

    return utterances[output_columns].copy()


utterances = create_utterances_table(wired_raw)

utterances.head()

,dataset,video_id,conversation_id,file_number,level,level_label,utterance_id,utterance_number,turn_id,utterance_in_turn,...,speaker,speaker_raw,speaker_role,text,n_words,include_geometry_primary,embedding_idx,notes,source_file,_original_dataframe_row
0,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_001,1,1,1,...,B,Speaker 2 (Genesis - child),partner,what's this,2,False,0,NaN,data/wired/wired_talia/wired_Talia_12.csv,5386
1,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_002,2,2,1,...,A,Speaker 1 (Talia Gershon),expert,yeah what do you think that is,7,True,1,NaN,data/wired/wired_talia/wired_Talia_12.csv,5387
2,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_003,3,3,1,...,B,Speaker 2 (Genesis - child),partner,fancy chandelier,2,False,2,NaN,data/wired/wired_talia/wired_Talia_12.csv,5388
3,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_004,4,4,1,...,A,Speaker 1 (Talia Gershon),expert,i think so too we jokingly call it the chandelier,10,True,3,NaN,data/wired/wired_talia/wired_Talia_12.csv,5389
4,wired,talia,wired_Talia_12,12,0,child,wired::wired_Talia_12::utterance_005,5,4,2,...,A,Speaker 1 (Talia Gershon),expert,that's real gold you know,5,True,4,NaN,data/wired/wired_talia/wired_Talia_12.csv,5390


In [11]:
from sentence_transformers import SentenceTransformer


EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())
print("Maximum sequence length:", embedding_model.max_seq_length)

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding dimension: 384
Maximum sequence length: 256


In [12]:
# Prepare and validate utterance texts
utterance_texts = utterances["text"].astype(str).tolist()

utterances["n_model_tokens"] = [
    len(
        embedding_model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )
    for text in utterance_texts
]

utterances["exceeds_model_limit"] = (
    utterances["n_model_tokens"]
    > embedding_model.max_seq_length
)

if utterances["exceeds_model_limit"].any():
    raise ValueError(
        "At least one utterance exceeds the model's token limit."
    )

# Generate one normalized embedding per utterance
E_utterances = embedding_model.encode(
    utterance_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

E_utterances = np.asarray(E_utterances, dtype=np.float32)

assert E_utterances.shape[0] == len(utterances)
assert np.isfinite(E_utterances).all()
assert np.allclose(
    np.linalg.norm(E_utterances, axis=1),
    1.0,
    atol=1e-5,
)

print("E_utterances shape:", E_utterances.shape)

Batches:   0%|          | 0/200 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Batches: 100%|██████████| 200/200 [00:07<00:00, 27.96it/s]

E_utterances shape: (6384, 384)


In [13]:
geometry_utterances = utterances.copy()

print("All utterances:", len(utterances))
print(
    "Primary geometry utterances:",
    len(geometry_utterances),
)
print(
    "Excluded short utterances:",
    len(utterances) - len(geometry_utterances),
)

All utterances: 6384
Primary geometry utterances: 6384
Excluded short utterances: 0


# Participation-ratio intuition

This synthetic illustration holds total variance constant while changing how evenly it is distributed across directions.

In [14]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)
n = 200

# Equal total variance, differently distributed across directions
low_eigenvalues  = np.array([9.0, 0.7, 0.3])
high_eigenvalues = np.array([10/3, 10/3, 10/3])

low = rng.normal(size=(n, 3)) * np.sqrt(low_eigenvalues)
high = rng.normal(size=(n, 3)) * np.sqrt(high_eigenvalues)

def theoretical_pr(eigenvalues):
    return eigenvalues.sum()**2 / np.square(eigenvalues).sum()

fig = plt.figure(figsize=(12, 5))

for i, (points, eigenvalues, label) in enumerate([
    (low, low_eigenvalues, "Low PR: variance concentrated"),
    (high, high_eigenvalues, "High PR: variance distributed"),
], start=1):

    ax = fig.add_subplot(1, 2, i, projection="3d")

    ax.scatter(
        points[:, 0],
        points[:, 1],
        points[:, 2],
        s=16,
        alpha=0.55,
    )

    ax.set_title(
        f"{label}\nPR = {theoretical_pr(eigenvalues):.2f}"
    )
    ax.set_xlabel("Direction 1")
    ax.set_ylabel("Direction 2")
    ax.set_zlabel("Direction 3")
    ax.set_xlim(-7, 7)
    ax.set_ylim(-7, 7)
    ax.set_zlim(-7, 7)
    ax.set_box_aspect((1, 1, 1))

plt.tight_layout()
plt.show()

## bias corrected PR

In [15]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [16]:
%pip install -q dimensionality


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [17]:
import numpy as np
import pandas as pd

from dimensionality import participation_ratio
from sklearn.preprocessing import normalize

GROUP_COLS = [
    "dataset",
    "video_id",
    "conversation_id",
    "level_label",
]

records = []

for group_values, group in geometry_utterances.groupby(
    GROUP_COLS,
    observed=True,
    sort=False,
):
    indices = group["embedding_idx"].astype(int).to_numpy()

    # Keep this normalization if your original PR used
    # unit-normalized embeddings.
    X = normalize(
        np.asarray(E_utterances[indices]),
        norm="l2",
        axis=1,
    )

    # Do not subtract X.mean(axis=0)
    result = participation_ratio(
        X,
        return_all=True,
    )

    record = dict(zip(GROUP_COLS, group_values))

    record.update({
        "n_utterances": len(X),
        "pr_naive_package": result["naive"],
        "pr_row_corrected": result["row"],
        "pr_both_corrected": result["both"],
    })

    records.append(record)

pr_corrected = pd.DataFrame(records)

display(
    pr_corrected.groupby(
        "level_label",
        observed=True,
    )[
        [
            "n_utterances",
            "pr_naive_package",
            "pr_row_corrected",
        ]
    ].mean()
)

,n_utterances,pr_naive_package,pr_row_corrected
level_label,,,
child,53.952381,19.803006,34.551041
expert,71.809524,27.881966,50.415688
graduate,56.238095,22.873298,42.916053
teenager,58.285714,20.747445,36.345679
undergraduate,63.714286,23.411450,40.528871


In [18]:
import statsmodels.formula.api as smf

gap_map = {
    "expert": 0,
    "graduate": 1,
    "undergraduate": 2,
    "teenager": 3,
    "child": 4,
}

pr_model_data = pr_corrected.copy()

pr_model_data["expertise_gap"] = (
    pr_model_data["level_label"].map(gap_map)
)

pr_model_data["match_id"] = (
    pr_model_data["dataset"].astype(str)
    + "::"
    + pr_model_data["video_id"].astype(str)
)

pr_model_data["pr_row_z"] = (
    pr_model_data["pr_row_corrected"]
    - pr_model_data["pr_row_corrected"].mean()
) / pr_model_data["pr_row_corrected"].std(ddof=0)

pr_model_data["expertise_gap_z"] = (
    pr_model_data["expertise_gap"]
    - pr_model_data["expertise_gap"].mean()
) / pr_model_data["expertise_gap"].std(ddof=0)

pr_row_model = smf.ols(
    "pr_row_z ~ expertise_gap_z + C(match_id)",
    data=pr_model_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": pr_model_data["match_id"]
    },
)

print(
    pr_row_model.summary().tables[1]
)

                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                           -0.2263   7.01e-16  -3.23e+14      0.000      -0.226      -0.226
C(match_id)[T.wired::blockchain]     0.4388   1.32e-15   3.32e+14      0.000       0.439       0.439
C(match_id)[T.wired::crispr]        -0.4014   9.16e-16  -4.38e+14      0.000      -0.401      -0.401
C(match_id)[T.wired::dimension]     -0.5284   8.22e-16  -6.43e+14      0.000      -0.528      -0.528
C(match_id)[T.wired::fractal]       -0.3292   8.34e-16  -3.95e+14      0.000      -0.329      -0.329
C(match_id)[T.wired::gravity]        0.7663   9.65e-16   7.94e+14      0.000       0.766       0.766
C(match_id)[T.wired::hacking]        1.0259      1e-15   1.03e+15      0.000       1.026       1.026
C(match_id)[T.wired::infinity]      -0.1915   7.27e-16  -2.64e+14      0.000      -0.192   

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 21, but rank is 1
  warnings.warn('covariance of constraints does not have full '


In [19]:
pr_row_model_t = smf.ols(
    "pr_row_z ~ expertise_gap_z + C(match_id)",
    data=pr_model_data,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": pr_model_data["match_id"],
        "use_correction": True,
        "df_correction": True,
    },
    use_t=True,
)

print(
    pr_row_model_t.summary().tables[1]
)

                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercept                           -0.2263   7.01e-16  -3.23e+14      0.000      -0.226      -0.226
C(match_id)[T.wired::blockchain]     0.4388   1.32e-15   3.32e+14      0.000       0.439       0.439
C(match_id)[T.wired::crispr]        -0.4014   9.16e-16  -4.38e+14      0.000      -0.401      -0.401
C(match_id)[T.wired::dimension]     -0.5284   8.22e-16  -6.43e+14      0.000      -0.528      -0.528
C(match_id)[T.wired::fractal]       -0.3292   8.34e-16  -3.95e+14      0.000      -0.329      -0.329
C(match_id)[T.wired::gravity]        0.7663   9.65e-16   7.94e+14      0.000       0.766       0.766
C(match_id)[T.wired::hacking]        1.0259      1e-15   1.03e+15      0.000       1.026       1.026
C(match_id)[T.wired::infinity]      -0.1915   7.27e-16  -2.64e+14      0.000      -0.192   

/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 21, but rank is 1
  warnings.warn('covariance of constraints does not have full '


Bias-corrected participation ratio decreased significantly as the expertise gap increased, β=−0.518, SE=0.113, 95% CI [−0.739,−0.296], indicating that expert-level conversations occupied a greater number of effective semantic directions even after correcting for differences in utterance count.

## how PR evolved. cumulative, rolling

In [20]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dimensionality import participation_ratio


# Primary settings
ROLLING_WINDOW = 12
ROLLING_STEP = 2
SHIFT_CONTEXT = 6

MIN_CUMULATIVE = 8

CUMULATIVE_PROGRESS = np.round(
    np.arange(0.20, 1.001, 0.05),
    2,
)


def row_corrected_pr(X):
    """
    Bias-corrected participation ratio for sampled rows.

    X must not be mean-centered. E_utterances is already
    unit-normalized, so no additional preprocessing is needed.
    """
    X = np.asarray(X, dtype=np.float64)

    if X.ndim != 2 or len(X) < 4:
        return np.nan

    result = participation_ratio(
        X,
        return_all=True,
    )

    return float(result["row"])


def centroid_cosine_shift(X_before, X_after):
    """
    Directional distance between the centroids before and after
    a possible semantic boundary.
    """
    before = np.mean(X_before, axis=0)
    after = np.mean(X_after, axis=0)

    denominator = (
        np.linalg.norm(before)
        * np.linalg.norm(after)
    )

    if denominator <= 1e-12:
        return np.nan

    similarity = np.clip(
        np.dot(before, after) / denominator,
        -1.0,
        1.0,
    )

    return float(1.0 - similarity)

In [21]:
cumulative_records = []
local_records = []
shift_records = []

GROUP_COLUMNS = [
    "dataset",
    "conversation_id",
]

for group_key, group in geometry_utterances.groupby(
    GROUP_COLUMNS,
    observed=True,
    sort=False,
):
    dataset, conversation_id = group_key

    group = (
        group
        .sort_values(
            "embedding_idx",
            kind="stable",
        )
        .reset_index(drop=True)
    )

    embedding_indices = (
        group["embedding_idx"]
        .to_numpy(dtype=int)
    )

    X = np.asarray(
        E_utterances[embedding_indices],
        dtype=np.float64,
    )

    n = len(X)

    metadata = {
        "dataset": dataset,
        "conversation_id": conversation_id,
        "n_utterances": n,
    }

    for column in [
        "video_id",
        "file_number",
        "level",
        "level_label",
    ]:
        if column in group.columns:
            value = group[column].iloc[0]

            if column == "level_label":
                value = str(value).lower()

            metadata[column] = value

    # ---------------------------------------------------------
    # 1. Cumulative dimensionality
    # ---------------------------------------------------------

    for target_progress in CUMULATIVE_PROGRESS:
        stop = int(
            np.ceil(target_progress * n)
        )

        if stop < MIN_CUMULATIVE:
            continue

        cumulative_records.append({
            **metadata,
            "progress_target": target_progress,
            "progress_actual": stop / n,
            "n_observed": stop,
            "cumulative_pr": row_corrected_pr(
                X[:stop]
            ),
        })

    # ---------------------------------------------------------
    # 2. Rolling/local dimensionality
    # ---------------------------------------------------------

    if n >= ROLLING_WINDOW:
        starts = list(
            range(
                0,
                n - ROLLING_WINDOW + 1,
                ROLLING_STEP,
            )
        )

        # Always include the final window
        final_start = n - ROLLING_WINDOW

        if starts[-1] != final_start:
            starts.append(final_start)

        for start in starts:
            stop = start + ROLLING_WINDOW

            midpoint = (
                start
                + (ROLLING_WINDOW - 1) / 2
            )

            local_records.append({
                **metadata,
                "window_start": start,
                "window_stop": stop,
                "progress": midpoint / (n - 1),
                "local_pr": row_corrected_pr(
                    X[start:stop]
                ),
            })

    # ---------------------------------------------------------
    # 3. Before/after centroid shift
    # ---------------------------------------------------------

    if n >= 2 * SHIFT_CONTEXT:
        boundaries = list(
            range(
                SHIFT_CONTEXT,
                n - SHIFT_CONTEXT + 1,
                ROLLING_STEP,
            )
        )

        final_boundary = n - SHIFT_CONTEXT

        if boundaries[-1] != final_boundary:
            boundaries.append(final_boundary)

        for boundary in boundaries:
            X_before = X[
                boundary - SHIFT_CONTEXT:
                boundary
            ]

            X_after = X[
                boundary:
                boundary + SHIFT_CONTEXT
            ]

            shift_records.append({
                **metadata,
                "boundary": boundary,
                "progress": (
                    boundary - 0.5
                ) / (n - 1),
                "centroid_shift": (
                    centroid_cosine_shift(
                        X_before,
                        X_after,
                    )
                ),
            })


cumulative_pr_time = pd.DataFrame(
    cumulative_records
)

local_pr_time = pd.DataFrame(
    local_records
)

centroid_shift_time = pd.DataFrame(
    shift_records
)

print(
    "Cumulative observations:",
    len(cumulative_pr_time),
)

print(
    "Local windows:",
    len(local_pr_time),
)

print(
    "Potential switch boundaries:",
    len(centroid_shift_time),
)

display(cumulative_pr_time.head())
display(local_pr_time.head())
display(centroid_shift_time.head())

Cumulative observations: 1737
Local windows: 2695
Potential switch boundaries: 2695


,dataset,conversation_id,n_utterances,video_id,file_number,level,level_label,progress_target,progress_actual,n_observed,cumulative_pr
0,wired,wired_Talia_12,27,talia,12,0,child,0.30,0.333333,9,28.272068
1,wired,wired_Talia_12,27,talia,12,0,child,0.35,0.370370,10,28.319875
2,wired,wired_Talia_12,27,talia,12,0,child,0.40,0.407407,11,25.742274
3,wired,wired_Talia_12,27,talia,12,0,child,0.45,0.481481,13,32.685983
4,wired,wired_Talia_12,27,talia,12,0,child,0.50,0.518519,14,37.029609


,dataset,conversation_id,n_utterances,video_id,file_number,level,level_label,window_start,window_stop,progress,local_pr
0,wired,wired_Talia_12,27,talia,12,0,child,0,12,0.211538,31.107675
1,wired,wired_Talia_12,27,talia,12,0,child,2,14,0.288462,39.767996
2,wired,wired_Talia_12,27,talia,12,0,child,4,16,0.365385,39.396549
3,wired,wired_Talia_12,27,talia,12,0,child,6,18,0.442308,34.757011
4,wired,wired_Talia_12,27,talia,12,0,child,8,20,0.519231,35.632175


,dataset,conversation_id,n_utterances,video_id,file_number,level,level_label,boundary,progress,centroid_shift
0,wired,wired_Talia_12,27,talia,12,0,child,6,0.211538,0.423626
1,wired,wired_Talia_12,27,talia,12,0,child,8,0.288462,0.448153
2,wired,wired_Talia_12,27,talia,12,0,child,10,0.365385,0.455338
3,wired,wired_Talia_12,27,talia,12,0,child,12,0.442308,0.502186
4,wired,wired_Talia_12,27,talia,12,0,child,14,0.519231,0.505248


In [22]:
display(
    cumulative_pr_time[
        "cumulative_pr"
    ].describe()
)

display(
    local_pr_time[
        "local_pr"
    ].describe()
)

print(
    "Nonfinite local estimates:",
    (~np.isfinite(
        local_pr_time["local_pr"]
    )).sum()
)

print(
    "Local estimates below 1:",
    (
        local_pr_time["local_pr"] < 1
    ).sum()
)

count    1737.000000
mean       36.848381
std        11.697944
min         5.888014
25%        28.728003
50%        35.831880
75%        44.005642
max        94.382656
Name: cumulative_pr, dtype: float64

count    2695.000000
mean       33.034975
std        15.197673
min         6.848253
25%        22.284499
50%        30.523450
75%        40.493947
max       129.716446
Name: local_pr, dtype: float64

Nonfinite local estimates: 0
Local estimates below 1: 0


In [23]:
#look at one convo
EXAMPLE_VIDEO = "time"
EXAMPLE_LEVEL = "expert"

selection = cumulative_pr_time[
    (
        cumulative_pr_time["video_id"]
        == EXAMPLE_VIDEO
    )
    &
    (
        cumulative_pr_time["level_label"]
        == EXAMPLE_LEVEL
    )
]

if selection.empty:
    raise ValueError(
        "No conversation matched the selected "
        "video and level."
    )

example_conversation = (
    selection["conversation_id"].iloc[0]
)

cum_example = cumulative_pr_time[
    cumulative_pr_time["conversation_id"]
    == example_conversation
]

local_example = local_pr_time[
    local_pr_time["conversation_id"]
    == example_conversation
]

shift_example = centroid_shift_time[
    centroid_shift_time["conversation_id"]
    == example_conversation
]

fig, axes = plt.subplots(
    3,
    1,
    figsize=(11, 9),
    sharex=True,
)

axes[0].plot(
    cum_example["progress_actual"],
    cum_example["cumulative_pr"],
    marker="o",
)

axes[0].set_ylabel(
    "Cumulative\ncorrected PR"
)

axes[0].set_title(
    f"Semantic dimensionality over time: "
    f"{example_conversation}"
)

axes[1].plot(
    local_example["progress"],
    local_example["local_pr"],
    marker="o",
    color="tab:orange",
)

axes[1].set_ylabel(
    "Rolling local\ncorrected PR"
)

axes[2].plot(
    shift_example["progress"],
    shift_example["centroid_shift"],
    marker="o",
    color="tab:green",
)

axes[2].set_ylabel(
    "Before–after\ncentroid shift"
)

axes[2].set_xlabel(
    "Normalized conversation progress"
)

for ax in axes:
    ax.grid(alpha=0.2)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## avg PR trajectories by convo level

In [24]:
# ============================================================
# Select complete matched videos with sufficiently long
# conversations at every expertise level
# ============================================================

MIN_TRAJECTORY_UTTERANCES = 20
CUMULATIVE_START = 0.40
MIN_BIN_COVERAGE = 0.0

conversation_meta = (
    cumulative_pr_time[
        [
            "dataset",
            "video_id",
            "conversation_id",
            "level_label",
            "n_utterances",
        ]
    ]
    .drop_duplicates()
)

eligible_matches = (
    conversation_meta
    .groupby(
        ["dataset", "video_id"],
        observed=True,
    )
    .agg(
        n_levels=(
            "level_label",
            "nunique",
        ),
        minimum_n=(
            "n_utterances",
            "min",
        ),
    )
    .reset_index()
)

eligible_matches = eligible_matches[
    (
        eligible_matches["n_levels"] == 5
    )
    &
    (
        eligible_matches["minimum_n"]
        >= MIN_TRAJECTORY_UTTERANCES
    )
][
    ["dataset", "video_id"]
]

print(
    "Eligible matched videos:",
    len(eligible_matches),
)


def keep_eligible_matches(df):
    return df.merge(
        eligible_matches,
        on=["dataset", "video_id"],
        how="inner",
        validate="many_to_one",
    )


cumulative_plot_data = keep_eligible_matches(
    cumulative_pr_time
)

local_plot_source = keep_eligible_matches(
    local_pr_time
)

shift_plot_source = keep_eligible_matches(
    centroid_shift_time
)

# At 40% of a 20-utterance conversation, at least
# eight utterances have been observed.
cumulative_plot_data = cumulative_plot_data[
    cumulative_plot_data["progress_target"]
    >= CUMULATIVE_START
].copy()

Eligible matched videos: 20


In [25]:
TIME_EDGES = np.linspace(0, 1, 11)


def conversation_time_bins(
    df,
    time_column,
    value_column,
):
    binned = df[
        [
            "dataset",
            "conversation_id",
            "level_label",
            time_column,
            value_column,
        ]
    ].dropna().copy()

    binned["time_bin"] = pd.cut(
        binned[time_column],
        bins=TIME_EDGES,
        labels=False,
        include_lowest=True,
    )

    binned = (
        binned
        .groupby(
            [
                "dataset",
                "conversation_id",
                "level_label",
                "time_bin",
            ],
            observed=True,
        )[value_column]
        .mean()
        .reset_index()
    )

    binned["progress"] = (
        binned["time_bin"] + 0.5
    ) / 10

    # Number of eligible conversations at each level
    total_conversations = (
        conversation_meta
        .merge(
            eligible_matches,
            on=["dataset", "video_id"],
            how="inner",
        )
        .groupby(
            "level_label",
            observed=True,
        )["conversation_id"]
        .nunique()
        .rename("total_conversations")
    )

    # Number contributing to each plotted bin
    coverage = (
        binned
        .groupby(
            ["level_label", "time_bin"],
            observed=True,
        )["conversation_id"]
        .nunique()
        .rename("n_conversations")
        .reset_index()
    )

    coverage = coverage.merge(
        total_conversations.reset_index(),
        on="level_label",
        how="left",
    )

    coverage["coverage"] = (
        coverage["n_conversations"]
        / coverage["total_conversations"]
    )

    binned = binned.merge(
        coverage,
        on=["level_label", "time_bin"],
        how="left",
    )

    return binned[
        binned["coverage"]
        >= MIN_BIN_COVERAGE
    ].copy()


local_plot_data = conversation_time_bins(
    local_plot_source,
    "progress",
    "local_pr",
)

shift_plot_data = conversation_time_bins(
    shift_plot_source,
    "progress",
    "centroid_shift",
)

In [26]:
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import Image, display


level_order = [
    "expert",
    "graduate",
    "undergraduate",
    "teenager",
    "child",
]

level_colors = expertise_palette(level_order)


def summarize_trajectory(
    df,
    time_column,
    value_column,
):
    return (
        df.dropna(
            subset=[
                "level_label",
                time_column,
                value_column,
            ]
        )
        .groupby(
            [
                "level_label",
                time_column,
            ],
            observed=True,
        )[value_column]
        .agg(
            mean="mean",
            sd="std",
            n="count",
        )
        .reset_index()
        .assign(
            se=lambda x:
                x["sd"] / np.sqrt(x["n"])
        )
    )


cumulative_summary = summarize_trajectory(
    cumulative_plot_data,
    "progress_target",
    "cumulative_pr",
)

local_summary = summarize_trajectory(
    local_plot_data,
    "progress",
    "local_pr",
)

shift_summary = summarize_trajectory(
    shift_plot_data,
    "progress",
    "centroid_shift",
)


plot_specs = [
    {
        "data": cumulative_summary,
        "time": "progress_target",
        "ylabel": "Cumulative corrected PR",
    },
    {
        "data": local_summary,
        "time": "progress",
        "ylabel": "Rolling local corrected PR",
    },
    {
        "data": shift_summary,
        "time": "progress",
        "ylabel": "Before–after centroid shift",
    },
]


fig, axes = plt.subplots(
    3,
    1,
    figsize=(12, 11),

    # The measures have different valid temporal ranges.
    sharex=False,
)


for ax, spec in zip(
    axes,
    plot_specs,
):
    summary = spec["data"]
    time_column = spec["time"]

    for level in level_order:
        level_data = (
            summary[
                summary["level_label"]
                .astype(str)
                .str.lower()
                == level
            ]
            .sort_values(time_column)
        )

        if level_data.empty:
            continue

        x = level_data[
            time_column
        ].to_numpy(dtype=float)

        mean = level_data[
            "mean"
        ].to_numpy(dtype=float)

        se = (
            level_data["se"]
            .fillna(0)
            .to_numpy(dtype=float)
        )

        color = level_colors[level]

        ax.plot(
            x,
            mean,
            marker="o",
            linewidth=2,
            markersize=4,
            color=color,
            label=level,
        )

        ax.fill_between(
            x,
            mean - se,
            mean + se,
            color=color,
            alpha=0.15,
        )

    ax.set_ylabel(spec["ylabel"])
    ax.set_xlabel(
        "Normalized conversation progress"
    )
    ax.grid(alpha=0.2)

    # Use the range where this measure actually exists.
    available_times = summary[
        time_column
    ].dropna()

    if len(available_times):
        left = max(
            0,
            available_times.min() - 0.03,
        )

        right = min(
            1,
            available_times.max() + 0.03,
        )

        ax.set_xlim(left, right)


axes[0].set_title(
    "Evolution of semantic dimensionality by expertise"
)

axes[0].legend(
    title="Expertise level",
    frameon=False,
)

plt.tight_layout()

output_path = (
    "semantic_dimensionality_trajectories.png"
)

fig.savefig(
    output_path,
    dpi=130,
    bbox_inches="tight",
    facecolor="white",
)

plt.close(fig)

display(
    Image(
        filename=output_path,
        width=1100,
    )
)

# Dual semantic-space and PR animation

In [30]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import textwrap
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from dimensionality import participation_ratio


# ------------------------------------------------------------
# Select conversation
# ------------------------------------------------------------
'''
levels: "child", "teenager","undergraduate", "graduate", "expert",
'''
VIDEO_ID = "gravity"
LEVEL_LABEL = "child"

MIN_PR_UTTERANCES = 12


selected_conversation = (
    geometry_utterances[
        (
            geometry_utterances["video_id"]
            .astype(str)
            == VIDEO_ID
        )
        &
        (
            geometry_utterances["level_label"]
            .astype(str)
            .str.lower()
            == LEVEL_LABEL
        )
    ]
    .sort_values(
        "embedding_idx",
        kind="stable",
    )
    .reset_index(drop=True)
)

if selected_conversation.empty:
    raise ValueError(
        "No conversation matched VIDEO_ID and LEVEL_LABEL."
    )

conversation_id = (
    selected_conversation[
        "conversation_id"
    ].iloc[0]
)

embedding_indices = (
    selected_conversation[
        "embedding_idx"
    ]
    .to_numpy(dtype=int)
)

X_animation = np.asarray(
    E_utterances[embedding_indices],
    dtype=np.float64,
)

n_animation_utterances = len(X_animation)

utterance_text = (
    selected_conversation["text"]
    .astype(str)
    .to_numpy()
)

if "speaker_role" in selected_conversation.columns:
    speaker_roles = (
        selected_conversation[
            "speaker_role"
        ]
        .astype(str)
        .str.lower()
        .to_numpy()
    )
else:
    speaker_roles = np.repeat(
        "speaker",
        n_animation_utterances,
    )


# ------------------------------------------------------------
# Fit PCA once to the complete conversation
# ------------------------------------------------------------

pca_animation = PCA(
    n_components=2,
    svd_solver="full",
)

X_animation_2d = pca_animation.fit_transform(
    X_animation
)

explained = (
    100
    * pca_animation.explained_variance_ratio_
)


# ------------------------------------------------------------
# Calculate full-space cumulative corrected PR
# ------------------------------------------------------------

def calculate_row_corrected_pr(X):
    if len(X) < 4:
        return np.nan

    result = participation_ratio(
        np.asarray(X, dtype=np.float64),
        return_all=True,
    )

    return float(result["row"])


cumulative_pr_animation = np.full(
    n_animation_utterances,
    np.nan,
)

for stop in range(
    MIN_PR_UTTERANCES,
    n_animation_utterances + 1,
):
    cumulative_pr_animation[
        stop - 1
    ] = calculate_row_corrected_pr(
        X_animation[:stop]
    )


print("Conversation:", conversation_id)
print("Utterances:", n_animation_utterances)
print(
    "PCA variance displayed:",
    f"{explained.sum():.1f}%",
)
print(
    "Finite PR estimates:",
    np.isfinite(
        cumulative_pr_animation
    ).sum(),
)

Conversation: wired_gravity_12
Utterances: 72
PCA variance displayed: 20.2%
Finite PR estimates: 61


In [31]:
from matplotlib.animation import (
    FuncAnimation,
    PillowWriter,
)
from matplotlib.lines import Line2D
from IPython.display import Image, display


# ------------------------------------------------------------
# Stable plotting ranges
# ------------------------------------------------------------

x_padding = max(
    0.10 * np.ptp(X_animation_2d[:, 0]),
    0.05,
)

y_padding = max(
    0.10 * np.ptp(X_animation_2d[:, 1]),
    0.05,
)

pca_xlim = (
    X_animation_2d[:, 0].min() - x_padding,
    X_animation_2d[:, 0].max() + x_padding,
)

pca_ylim = (
    X_animation_2d[:, 1].min() - y_padding,
    X_animation_2d[:, 1].max() + y_padding,
)

finite_pr = cumulative_pr_animation[
    np.isfinite(cumulative_pr_animation)
]

if len(finite_pr):
    pr_padding = max(
        0.10 * np.ptp(finite_pr),
        1.0,
    )

    pr_ylim = (
        finite_pr.min() - pr_padding,
        finite_pr.max() + pr_padding,
    )
else:
    pr_ylim = (0, 1)


role_colors = {
    "expert": "tab:blue",
    "partner": "tab:orange",
    "speaker": "tab:gray",
}

point_colors = np.array([
    role_colors.get(
        role,
        "tab:gray",
    )
    for role in speaker_roles
])


# ------------------------------------------------------------
# Construct figure
# ------------------------------------------------------------

fig, (
    ax_space,
    ax_pr,
) = plt.subplots(
    1,
    2,
    figsize=(14, 6),
)

fig.suptitle(
    f"Semantic-space expansion: {conversation_id}",
    fontsize=14,
)

# Left: PCA semantic trajectory
trajectory_line, = ax_space.plot(
    [],
    [],
    color="0.60",
    linewidth=1,
    alpha=0.7,
    zorder=1,
)

past_points = ax_space.scatter(
    [],
    [],
    s=42,
    alpha=0.75,
    zorder=2,
)

current_point = ax_space.scatter(
    [],
    [],
    s=150,
    facecolors="none",
    edgecolors="black",
    linewidths=2,
    zorder=3,
)

time_label = ax_space.text(
    0.02,
    0.98,
    "",
    transform=ax_space.transAxes,
    ha="left",
    va="top",
)

ax_space.set_xlim(*pca_xlim)
ax_space.set_ylim(*pca_ylim)

ax_space.set_xlabel(
    f"PC1 ({explained[0]:.1f}% displayed variance)"
)

ax_space.set_ylabel(
    f"PC2 ({explained[1]:.1f}% displayed variance)"
)

ax_space.set_title(
    "Cumulative 2D PCA view"
)

ax_space.grid(alpha=0.2)


# Only include roles actually present
legend_handles = []

for role, label in [
    ("expert", "Expert speaker"),
    ("partner", "Conversation partner"),
    ("speaker", "Speaker"),
]:
    if role in set(speaker_roles):
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="none",
                markerfacecolor=role_colors[role],
                markeredgecolor="none",
                markersize=7,
                label=label,
            )
        )

if legend_handles:
    ax_space.legend(
        handles=legend_handles,
        frameon=False,
        loc="lower right",
    )


# Right: full-dimensional cumulative PR
pr_line, = ax_pr.plot(
    [],
    [],
    color="tab:purple",
    linewidth=2.5,
)

pr_current, = ax_pr.plot(
    [],
    [],
    marker="o",
    color="tab:purple",
    markersize=8,
)

ax_pr.axvline(
    MIN_PR_UTTERANCES,
    color="0.65",
    linestyle="--",
    linewidth=1,
)

ax_pr.text(
    MIN_PR_UTTERANCES,
    pr_ylim[1],
    " PR begins",
    color="0.40",
    ha="left",
    va="top",
)

ax_pr.set_xlim(
    1,
    n_animation_utterances,
)

ax_pr.set_ylim(*pr_ylim)

ax_pr.set_xlabel(
    "Utterances observed"
)

ax_pr.set_ylabel(
    "Cumulative row-corrected PR"
)

ax_pr.set_title(
    "Full 384-dimensional estimate"
)

ax_pr.grid(alpha=0.2)


# Current utterance caption
caption = fig.text(
    0.5,
    0.025,
    "",
    ha="center",
    va="bottom",
    fontsize=10,
)

plt.tight_layout(
    rect=[0, 0.13, 1, 0.94]
)


# ------------------------------------------------------------
# Animation update
# ------------------------------------------------------------

def update_animation(frame):
    stop = frame + 1

    visible_points = (
        X_animation_2d[:stop]
    )

    past_points.set_offsets(
        visible_points
    )

    past_points.set_facecolors(
        point_colors[:stop]
    )

    trajectory_line.set_data(
        visible_points[:, 0],
        visible_points[:, 1],
    )

    current_point.set_offsets(
        X_animation_2d[
            frame:
            frame + 1
        ]
    )

    current_point.set_edgecolors(
        point_colors[frame]
    )

    time_label.set_text(
        f"Utterance {stop}/{n_animation_utterances}"
    )

    visible_pr = (
        cumulative_pr_animation[:stop]
    )

    valid = np.isfinite(
        visible_pr
    )

    pr_x = np.arange(
        1,
        stop + 1,
    )[valid]

    pr_y = visible_pr[valid]

    pr_line.set_data(
        pr_x,
        pr_y,
    )

    if np.isfinite(
        cumulative_pr_animation[frame]
    ):
        pr_current.set_data(
            [stop],
            [
                cumulative_pr_animation[
                    frame
                ]
            ],
        )
    else:
        pr_current.set_data(
            [],
            [],
        )

    role = speaker_roles[frame]

    caption.set_text(
        textwrap.fill(
            (
                f"{role.capitalize()}: "
                f"{utterance_text[frame]}"
            ),
            width=105,
        )
    )

    return (
        trajectory_line,
        past_points,
        current_point,
        time_label,
        pr_line,
        pr_current,
        caption,
    )


animation = FuncAnimation(
    fig,
    update_animation,
    frames=n_animation_utterances,
    interval=300,
    repeat=False,
    blit=False,
)


# ------------------------------------------------------------
# Save and display
# ------------------------------------------------------------

safe_conversation_id = re.sub(
    r"[^a-zA-Z0-9_-]+",
    "-",
    str(conversation_id),
)

gif_path = (
    f"semantic-pr-animation-"
    f"{safe_conversation_id}.gif"
)

animation.save(
    gif_path,
    writer=PillowWriter(fps=3),
    dpi=100,
)

plt.close(fig)

display(
    Image(
        filename=gif_path,
        width=1100,
    )
)